In [1]:
from pathlib import Path
import pandas as pd

BASE_DIR = Path(".") if Path("data").exists() else Path("..")
df = pd.read_csv(BASE_DIR / "data/data.csv", delimiter=";", decimal=".")
df.head()

,participant,condition,order,event,target_x,target_y,mean_x,mean_y,dist_x,dist_y,euclidean,sd_x,sd_y,rmss2s,lux
0,1,tobii,0,1,110,90,137.539758,108.080387,27.539758,18.080387,34.038937,8.714941,7.413912,7.519536,570
1,1,tobii,0,2,960,140,962.664858,142.831499,6.192006,10.431466,12.812091,6.905423,12.152017,9.736340,570
2,1,tobii,0,3,1810,90,1811.862020,111.039403,7.698739,21.039403,23.198701,8.814978,8.602357,7.274915,570
3,1,tobii,0,4,510,340,510.214306,317.034811,9.828926,22.965189,26.193878,14.136795,8.895412,9.016044,570
4,1,tobii,0,5,1410,340,1436.747614,335.238041,26.780094,8.563001,29.188782,9.205815,9.473244,9.409998,570


In [2]:
# Calculate the mean for dist_x and dist_y for the conditions tobii, tobii_filtered, uniteye, uniteye_filtered
mean_df = df.groupby('condition')[['dist_x', 'dist_y', 'euclidean', 'sd_x', 'sd_y', 'rmss2s']].mean()
mean_df

,dist_x,dist_y,euclidean,sd_x,sd_y,rmss2s
condition,,,,,,
tobii,22.597374,28.425218,39.798871,13.280550,12.928298,12.299467
uniteye,147.608274,141.719177,226.003278,65.625365,64.682136,113.990213
uniteye_filtered,141.788854,133.071396,213.538731,29.345580,27.602975,14.663301


In [3]:
from math import degrees, atan2


def px_to_deg(px: int, px_inch: int, eye_screen_distance_mm: int) -> float:
    return degrees(atan2(px / px_inch * 25.4, eye_screen_distance_mm))


display_size_in_cm = [34.5, 19.5]
pixel_per_inch = 141
pixel_per_cm = pixel_per_inch / 2.54

# Translate the mean values from pixel to cm
mean_df['dist_x_cm'] = mean_df['dist_x'] / pixel_per_cm
mean_df['dist_y_cm'] = mean_df['dist_y'] / pixel_per_cm
mean_df['rmss2s_cm'] = mean_df['rmss2s'] / pixel_per_cm
mean_df['euclidean_cm'] = mean_df['euclidean'] / pixel_per_cm
mean_df['sd_x_cm'] = mean_df['sd_x'] / pixel_per_cm
mean_df['sd_y_cm'] = mean_df['sd_y'] / pixel_per_cm

mean_df['euclidean_vd'] = mean_df['euclidean'].apply(lambda x: px_to_deg(x, pixel_per_inch, 600))
 
mean_df

,dist_x,dist_y,euclidean,sd_x,sd_y,rmss2s,dist_x_cm,dist_y_cm,rmss2s_cm,euclidean_cm,sd_x_cm,sd_y_cm,euclidean_vd
condition,,,,,,,,,,,,,
tobii,22.597374,28.425218,39.798871,13.280550,12.928298,12.299467,0.407073,0.512057,0.221565,0.716944,0.239238,0.232893,0.684599
uniteye,147.608274,141.719177,226.003278,65.625365,64.682136,113.990213,2.659043,2.552955,2.053441,4.071265,1.182187,1.165196,3.881821
uniteye_filtered,141.788854,133.071396,213.538731,29.345580,27.602975,14.663301,2.554211,2.397173,0.264147,3.846726,0.528637,0.497245,3.668332


In [4]:
# Calculate in how many areas the display can be divided with the given mean values
mean_df['areas_size_x'] = 2 * (mean_df['dist_x_cm'] + mean_df['sd_x_cm'])
mean_df['areas_size_y'] = 2 * (mean_df['dist_y_cm'] + mean_df['sd_y_cm'])

mean_df['areas_x'] = display_size_in_cm[0] / mean_df['areas_size_x']
mean_df['areas_y'] = display_size_in_cm[1] / mean_df['areas_size_y']

# round the area values to integers
mean_df['areas_x'] = mean_df['areas_x'].round(0).astype(int)
mean_df['areas_y'] = mean_df['areas_y'].round(0).astype(int)

mean_df['areas_size_x'] = mean_df['areas_size_x'].round(2)
mean_df['areas_size_y'] = mean_df['areas_size_y'].round(2)

# calculate the total number of areas
mean_df['number_of_areas'] = mean_df['areas_x'] * mean_df['areas_y']

mean_df

,dist_x,dist_y,euclidean,sd_x,sd_y,rmss2s,dist_x_cm,dist_y_cm,rmss2s_cm,euclidean_cm,sd_x_cm,sd_y_cm,euclidean_vd,areas_size_x,areas_size_y,areas_x,areas_y,number_of_areas
condition,,,,,,,,,,,,,,,,,,
tobii,22.597374,28.425218,39.798871,13.280550,12.928298,12.299467,0.407073,0.512057,0.221565,0.716944,0.239238,0.232893,0.684599,1.29,1.49,27,13,351
uniteye,147.608274,141.719177,226.003278,65.625365,64.682136,113.990213,2.659043,2.552955,2.053441,4.071265,1.182187,1.165196,3.881821,7.68,7.44,4,3,12
uniteye_filtered,141.788854,133.071396,213.538731,29.345580,27.602975,14.663301,2.554211,2.397173,0.264147,3.846726,0.528637,0.497245,3.668332,6.17,5.79,6,3,18


In [5]:
mean_df.to_csv(BASE_DIR / "aois_15inch.csv", sep=";", decimal=".")